# Notebook teatro AI

## Importar librerias

In [14]:
import pandas as pd
from os import listdir
import operator

Importar datos

In [15]:

path = 'data_example'
for file in listdir(path):
    if 'ipynb_checkpoints' not in file:
        data = pd.read_csv(path + '/' +  file) 
data.dropna(inplace= True)

In [16]:
p_number = {
1 : 'Quina edat tens?',
2 : 'Gènere',
3 : 'Quin tipus de formació acadèmica tens?',
4 : 'Et consideres una persona feliç?',
5 : 'Has sofert mai una depressió?',
6 : 'Creus en l’amor a primera vista?',
7 : 'Quantes parelles estables creus que has tingut?',
8 : 'T’agrada més la muntanya o la platja?',
9 : 'Et consideres una persona puntual?',
10 : 'Actualment fas donatius a algun tipus d’entitat social?',
11 : 'Acostumes a donar diners a captaires?',
12 : "Creus que l'us de la violència pot ser justificada?",
13 : 'Creus que la fi pot justificar els mitjans?',
14 : 'Quin barri barceloní dels següents preferiries viure?',
15 : 'Aturaries a un desconegut que s’anés a treure la vida?',
16 : 'Creus que pagues massa impostos?',
17 : 'Políticament, et consideres una persona de...',
18 : "Al llarg d' una setmana et mastubes:",
19 : 'Prefereixes que el teu cap superior de feina sigui...',
20 : 'Prioritzes el plaer als diners?'}

In [17]:
# Data correction
data[p_number[7]].replace({"4 o més": 4}, inplace=True)
data[p_number[7]] = data[p_number[7]].astype(int)

data[p_number[18]].replace({"3 o més": 3}, inplace=True)
data[p_number[18]] = data[p_number[18]].astype(int)

### Frase 1: Las personas que creen en el amor a primera vista prefieren tener un jefe de un género {distinto} al suyo.

Código:
Miraremos de las respuestas afirmativas de la pregunta 6 si el género coincide con el de la pregunta 19.


In [18]:
# gente que cree en el amor a primera vista
yes_data = data[data[p_number[6]] == 'Sí']
#yes count es gente que prefiere el mismo genero que ellos
yes_count = len(yes_data[yes_data[p_number[2]] == yes_data[p_number[19]]])
no_count = len(yes_data[yes_data[p_number[2]] != yes_data[p_number[19]]])
    
print(f'yes count {yes_count}')
print(f'no count {no_count}')

text = 'igual' if yes_count > no_count else 'diferent'

print(f'Les persones que creuen en l\'amor a primera vista prefereixen tenir un cap amb un gènere {text} al seu.')

yes count 5
no count 4
Les persones que creuen en l'amor a primera vista prefereixen tenir un cap amb un gènere igual al seu.


### Frase 2 : La gente joven (<30) acostumbran a dar {más/menos} dinero a necesitados y ONG’s



Código:
Miraremos si las respuestas 1 y 2 de la pregunta 1 si hay mas respuestas afirmativas en la pregunta 11


In [19]:
# Young peple
young_data = data[data[p_number[1]].str.contains('20')]
# Adults
old_data = data[data[p_number[1]].str.contains('20') == False]
young_count, old_count = 0, 0

young_count = young_data[p_number[11]].str.contains('Sí').value_counts().loc[True]
old_count = old_data[p_number[11]].str.contains('Sí').value_counts().loc[True] 

young = young_count/young_data.shape[0]
old = old_count/old_data.shape[0]

print('% Young donors: ' + str(young))
print('% Old donors: ' + str(old))


text = 'més' if young > old else 'menys'

print(f'La gent de menys de 30 anys acostuma a donar {text} diners a necessitats y ONG\'s.')

% Young donors: 0.625
% Old donors: 0.5
La gent de menys de 30 anys acostuma a donar més diners a necessitats y ONG's.


### Frase3: Las personas impuntuales prefieren {playa/montaña} y tienen {más/menos} parejas estables.

In [20]:
# puntuales
puntuales_data = data[data[p_number[9]] == 'Sí']
# impuntuales
impuntuales_data = data[data[p_number[9]] == 'No']

#Playa/montaña comparacion entre los impuntuales
beach_count = len(impuntuales_data[p_number[8]].str.contains('latja'))
mount_count = len(impuntuales_data[p_number[8]].str.contains('latja') == False)   

# Comparacion de parejas estables entre puntuales e impuntuales
puntuales_count, impuntuales_count = 0, 0

for number, count in puntuales_data[p_number[7]].value_counts().to_dict().items():
    puntuales_count += number*count
    
for number, count in impuntuales_data[p_number[7]].value_counts().to_dict().items():
    impuntuales_count += number*count  

puntuales = puntuales_count/puntuales_data.shape[0]
impuntuales =  impuntuales_count/impuntuales_data.shape[0]

print('Media parejas estables de los puntuales: ' + str(puntuales))
print('Media parejas estables de los impuntuales: ' + str(impuntuales))

text1 = 'platja' if beach_count > mount_count else 'muntanya'
text2 = 'més' if impuntuales > puntuales else 'menys'

print(f'Les persones impuntuals prefereixen la {text1} i tenen {text2} parelles estables.')

Media parejas estables de los puntuales: 2.0
Media parejas estables de los impuntuales: 3.272727272727273
Les persones impuntuals prefereixen la muntanya i tenen més parelles estables.


### Frase 4 : Las personas {impuntuales} se masturban más.

Código:
El que más tenga de la respuesta 9. Separaremos las que ponga si o no en la pregunta 9, miraremos cuales tiene mayores resultados en la pregnta 18, y cambiara el texto a puntuales o impuntuales.



In [21]:
# gente puntual
yes_data = data[data[p_number[9]] == 'Sí']
# gente no puntual
no_data = data[data[p_number[9]] == 'No']
yes_count, no_count = 0, 0

for number, count in no_data[p_number[18]].value_counts().to_dict().items():
    no_count += number*count
    
for number, count in yes_data[p_number[18]].value_counts().to_dict().items():
    yes_count += number*count
    
print(f'yes count {yes_count}')
print(f'no count {no_count}')

text = 'puntuals' if yes_count > no_count else 'impuntuals'

print(f'Les persones {text} es masturben menys.')

yes count 11
no count 23
Les persones impuntuals es masturben menys.



### Frase5: Las personas que políticamente se consideran {} prefieren la {playa/montaña} y tienden a sufrir más casos de depresión.
Código:
Compararemos la pregunta 8 (playa o montaña) con el que más número de personas hayan sufrido depresión (pregunta 5).



In [49]:
partidos = data[p_number[17]].unique()

partido_dep = {}
for partido in partidos:
    data_partido = data[data[p_number[17]] == partido]
    partido_dep[partido] = data_partido[data_partido[p_number[5]] == 'Sí'].shape[0] / data_partido.shape[0]
 
partido = max(partido_dep, key=partido_dep.get)
data_partido = data[data[p_number[17]] == partido]
print(partido_dep)
   
# gente con depresion
yes_data = data_partido[data_partido[p_number[5]] == 'Sí']
#yes count es gente que prefiere la playa

yes_count = len(yes_data[yes_data[p_number[8]].str.contains('latja')])
no_count = len(yes_data[yes_data[p_number[8]].str.contains('latja') == False])   

  
print(f'playa count {yes_count},{yes_count/data_partido.shape[0]} %')
print(f'no playa count {no_count}, {no_count/data_partido.shape[0]}')

text = 'platja' if yes_count > no_count else 'muntanya'

print(f'Les persones que políticament es consideren {partido} prefereixen la {text} i acostumen a tenir més casos de depressió.')


{'Centre esquerre': 0.6, 'No em defineixo': 0.0, 'Esquerres': 0.5714285714285714, 'Centre': 0.0, 'Centre dreta': 0.5, 'Dretes': 1.0}
playa count 1,1.0 %
no playa count 0, 0.0
Les persones que políticament es consideren Dretes prefereixen la platja i acostumen a tenir més casos de depressió.
